In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os
import pickle
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

In [8]:
BASE = "/content/drive/MyDrive/MultiModalProject/Flickr8k"

TRAIN_CSV = os.path.join(BASE, "dataset_caption_train.csv")
VAL_CSV   = os.path.join(BASE, "dataset_caption_val.csv")
TEST_CSV  = os.path.join(BASE, "dataset_caption_test.csv")

TRAIN_FEATURES = os.path.join(BASE, "efficient_train.npy")
VAL_FEATURES   = os.path.join(BASE, "efficient_val.npy")
TEST_FEATURES  = os.path.join(BASE, "efficient_test.npy")

MODEL_DIR = os.path.join(BASE, "models", "LSTM")
os.makedirs(MODEL_DIR, exist_ok=True)

In [11]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(29112, 2)
(3238, 2)
(8095, 2)


In [12]:
train_features = np.load(
    TRAIN_FEATURES,
    allow_pickle=True
).item()

val_features = np.load(
    VAL_FEATURES,
    allow_pickle=True
).item()

test_features = np.load(
    TEST_FEATURES,
    allow_pickle=True
).item()

print(len(train_features))
print(len(val_features))
print(len(test_features))

5824
648
1619


In [6]:
counter = Counter()

for caption in train_df["caption"]:

    words = caption.split()

    # Ignore special tokens
    words = [
        w for w in words
        if w not in ["<start>", "<end>"]
    ]

    counter.update(words)

vocab = {
    "<pad>":0,
    "<start>":1,
    "<end>":2,
    "<unk>":3
}

index = 4

for word, count in counter.items():

    if count >= 2:
        vocab[word] = index
        index += 1

idx2word = {
    v:k
    for k,v in vocab.items()
}

print(len(vocab))
print(vocab["<start>"])
print(vocab["<end>"])

4454
1
2


In [7]:
with open(
    os.path.join(MODEL_DIR,"vocab.pkl"),
    "wb"
) as f:

    pickle.dump(vocab,f)

In [8]:
def encode_caption(caption):

    words = caption.split()

    encoded = [
        vocab["<start>"]
    ]

    for word in words:

        if word in ["<start>","<end>"]:
            continue

        encoded.append(
            vocab.get(
                word,
                vocab["<unk>"]
            )
        )

    encoded.append(
        vocab["<end>"]
    )

    return torch.tensor(encoded)

In [9]:
train_df["encoded"] = train_df["caption"].apply(
    encode_caption
)

val_df["encoded"] = val_df["caption"].apply(
    encode_caption
)

test_df["encoded"] = test_df["caption"].apply(
    encode_caption
)

In [10]:
print(vocab["<start>"])
print(vocab["<end>"])

print(train_df.iloc[0]["encoded"])

1
2
tensor([ 1,  4,  5,  6,  4,  7,  8,  9, 10, 11,  4, 12, 13, 14,  6, 15,  3, 16,
         2])


In [11]:
class CaptionDataset(Dataset):

    def __init__(
        self,
        dataframe,
        feature_dict
    ):

        self.df = dataframe.reset_index(drop=True)
        self.features = feature_dict

    def __len__(self):

        return len(self.df)

    def __getitem__(self,idx):

        row = self.df.iloc[idx]

        image = row["image"]

        feature = torch.tensor(
            self.features[image]
        ).float()

        caption = row["encoded"]

        return feature, caption

In [12]:
def collate_fn(batch):

    images = []
    captions = []

    for img, cap in batch:

        images.append(img)
        captions.append(cap)

    images = torch.stack(images)

    captions = pad_sequence(
        captions,
        batch_first=True,
        padding_value=vocab["<pad>"]
    )

    return images, captions

In [13]:
train_loader = DataLoader(
    CaptionDataset(
        train_df,
        train_features
    ),
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    CaptionDataset(
        val_df,
        val_features
    ),
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    CaptionDataset(
        test_df,
        test_features
    ),
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [14]:
images, captions = next(iter(train_loader))

print(images.shape)
print(captions.shape)

print(captions[0])

torch.Size([32, 49, 1280])
torch.Size([32, 26])
tensor([   1,    4,   74,   17,   30,   74,  204,   32,   35,   36, 3544,   13,
         139,   36,   24,  335,    6,   24,  381,    2,    0,    0,    0,    0,
           0,    0])


In [15]:
class BahdanauAttention(nn.Module):

    def __init__(self, encoder_dim, decoder_dim, attention_dim):

        super().__init__()

        self.encoder_att = nn.Linear(
            encoder_dim,
            attention_dim
        )

        self.decoder_att = nn.Linear(
            decoder_dim,
            attention_dim
        )

        self.full_att = nn.Linear(
            attention_dim,
            1
        )

        self.relu = nn.ReLU()

        self.softmax = nn.Softmax(dim=1)

    def forward(self, encoder_out, decoder_hidden):

        att1 = self.encoder_att(encoder_out)

        att2 = self.decoder_att(
            decoder_hidden
        ).unsqueeze(1)

        energy = self.full_att(
            self.relu(att1 + att2)
        ).squeeze(2)

        alpha = self.softmax(energy)

        context = (
            encoder_out *
            alpha.unsqueeze(2)
        ).sum(dim=1)

        return context, alpha

In [16]:
class AttentionLSTM(nn.Module):

    def __init__(
        self,
        vocab_size,
        encoder_dim=1280,
        embed_dim=256,
        decoder_dim=512,
        attention_dim=512,
        dropout=0.5
    ):

        super().__init__()

        self.decoder_dim = decoder_dim

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.attention = BahdanauAttention(
            encoder_dim,
            decoder_dim,
            attention_dim
        )

        self.lstm = nn.LSTMCell(
            embed_dim + encoder_dim,
            decoder_dim
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            decoder_dim,
            vocab_size
        )

    def forward(self, features, captions):

        batch_size = features.size(0)

        seq_len = captions.size(1)

        embeddings = self.embedding(captions)

        h = torch.zeros(
            batch_size,
            self.decoder_dim,
            device=features.device
        )

        c = torch.zeros_like(h)

        outputs = []

        for t in range(seq_len - 1):

            context, _ = self.attention(
                features,
                h
            )

            lstm_input = torch.cat(
                (
                    embeddings[:, t, :],
                    context
                ),
                dim=1
            )

            h, c = self.lstm(
                lstm_input,
                (h, c)
            )

            out = self.fc(
                self.dropout(h)
            )

            outputs.append(out)

        outputs = torch.stack(
            outputs,
            dim=1
        )

        return outputs

In [17]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = AttentionLSTM(
    vocab_size=len(vocab)
).to(device)

criterion = nn.CrossEntropyLoss(
    ignore_index=vocab["<pad>"]
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [18]:
images, captions = next(iter(train_loader))

images = images.to(device)
captions = captions.to(device)

outputs = model(
    images,
    captions
)

print(outputs.shape)

torch.Size([32, 22, 4454])


In [19]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0

    for images, captions in loader:

        images = images.to(device)
        captions = captions.to(device)

        optimizer.zero_grad()

        outputs = model(
            images,
            captions
        )

        loss = criterion(
            outputs.reshape(
                -1,
                outputs.size(-1)
            ),
            captions[:,1:].reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [20]:
def validate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0

    with torch.no_grad():

        for images, captions in loader:

            images = images.to(device)
            captions = captions.to(device)

            outputs = model(
                images,
                captions
            )

            loss = criterion(
                outputs.reshape(
                    -1,
                    outputs.size(-1)
                ),
                captions[:,1:].reshape(-1)
            )

            total_loss += loss.item()

    return total_loss / len(loader)

In [21]:
best_loss = float("inf")

patience = 5

counter = 0

num_epochs = 30

for epoch in range(num_epochs):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss = validate(
        model,
        val_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch+1:02d}"
        f" | Train {train_loss:.4f}"
        f" | Val {val_loss:.4f}"
    )

    if val_loss < best_loss:

        best_loss = val_loss

        counter = 0

        torch.save(
            model.state_dict(),
            os.path.join(
                MODEL_DIR,
                "lstm_attn.pth"
            )
        )

        print("✅ Best model saved")

    else:

        counter += 1

        print(
            f"Patience {counter}/{patience}"
        )

        if counter >= patience:

            print("🛑 Early stopping")

            break

Epoch 01 | Train 3.7699 | Val 3.1418
✅ Best model saved
Epoch 02 | Train 3.0069 | Val 2.9327
✅ Best model saved
Epoch 03 | Train 2.6936 | Val 2.8693
✅ Best model saved
Epoch 04 | Train 2.4611 | Val 2.8489
✅ Best model saved
Epoch 05 | Train 2.2684 | Val 2.8451
✅ Best model saved
Epoch 06 | Train 2.1058 | Val 2.8788
Patience 1/5
Epoch 07 | Train 1.9655 | Val 2.8906
Patience 2/5
Epoch 08 | Train 1.8425 | Val 2.9220
Patience 3/5
Epoch 09 | Train 1.7321 | Val 2.9576
Patience 4/5
Epoch 10 | Train 1.6363 | Val 3.0038
Patience 5/5
🛑 Early stopping


In [22]:
ckpt = torch.load(
    os.path.join(
        MODEL_DIR,
        "lstm_attn.pth"
    ),
    map_location="cpu"
)

print(
    ckpt["embedding.weight"].shape
)

print(len(vocab))

torch.Size([4454, 256])
4454


In [23]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = AttentionLSTM(
    vocab_size=len(vocab)
).to(device)

model.load_state_dict(
    torch.load(
        os.path.join(
            MODEL_DIR,
            "lstm_attn.pth"
        ),
        map_location=device
    )
)

model.eval()

print("✅ Model Loaded")

✅ Model Loaded


In [9]:
def generate_caption(
    model,
    feature,
    vocab,
    idx2word,
    max_len=30
):

    model.eval()

    with torch.no_grad():

        feature = torch.tensor(
            feature
        ).float().unsqueeze(0).to(device)

        h = torch.zeros(
            1,
            model.decoder_dim,
            device=device
        )

        c = torch.zeros_like(h)

        word = torch.tensor(
            [vocab["<start>"]],
            device=device
        )

        sentence = []

        for _ in range(max_len):

            embedding = model.embedding(word)

            context, _ = model.attention(
                feature,
                h
            )

            lstm_input = torch.cat(
                (
                    embedding.squeeze(1),
                    context
                ),
                dim=1
            )

            h, c = model.lstm(
                lstm_input,
                (h, c)
            )

            out = model.fc(h)

            pred = out.argmax(1).item()

            if pred == vocab["<end>"]:
                break

            sentence.append(
                idx2word.get(pred, "<unk>")
            )

            word = torch.tensor(
                [pred],
                device=device
            )

        return " ".join(sentence)

In [10]:
import matplotlib.pyplot as plt
from PIL import Image

model.load_state_dict(
    torch.load(
        os.path.join(MODEL_DIR, "lstm_attn.pth"),
        map_location=device
    )
)

model.eval()

row = test_df.sample(1).iloc[0]

img_name = row["image"]

caption = row["caption"]

feature = torch.tensor(
    test_features[img_name]
).float().unsqueeze(0).to(device)

print(img_name)
print(caption)
print(feature.shape)

img = Image.open(
    os.path.join(BASE, "Images", img_name)
)

plt.imshow(img)
plt.axis("off")
plt.show()

NameError: name 'model' is not defined

In [26]:
def generate_caption(model, feature, vocab, idx2word, max_len=30):

    model.eval()

    feature = feature.to(device)

    h = torch.zeros(
        1,
        model.decoder_dim,
        device=device
    )

    c = torch.zeros_like(h)

    word = torch.tensor(
        [vocab["<start>"]],
        device=device
    )

    result = []

    with torch.no_grad():

        for _ in range(max_len):

            emb = model.embedding(word)

            context, _ = model.attention(
                feature,
                h
            )

            lstm_input = torch.cat(
                [
                    emb.squeeze(1),
                    context
                ],
                dim=1
            )

            h, c = model.lstm(
                lstm_input,
                (h, c)
            )

            out = model.fc(h)

            pred = out.argmax(1).item()

            token = idx2word[pred]

            if token == "<end>":
                break

            if token not in ["<start>", "<pad>"]:
                result.append(token)

            word = torch.tensor(
                [[pred]],
                device=device
            )

    return " ".join(result)

In [27]:
prediction = generate_caption(
    model,
    feature,
    vocab,
    idx2word
)

print("REAL:")
print(caption)

print()

print("PRED:")
print(prediction)

REAL:
<start> group of ballerinas in white tutus get ready to dance <end>

PRED:
a group of people are watching a man in a white shirt and black pants is standing on a trampoline


In [3]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=5fa66b746f3ced59546e8bb0fb83156758da49a2b3f4d92bfc86b74162f072c3
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [1]:
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [4]:
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from tqdm import tqdm

In [5]:
from collections import defaultdict

references_dict = defaultdict(list)

for _, row in test_df.iterrows():

    gt = row["caption"]

    gt = gt.replace("<start>", "").replace("<end>", "").strip()

    references_dict[row["image"]].append(gt.split())

print("Number of test images:", len(references_dict))

NameError: name 'test_df' is not defined

In [ ]:
from tqdm import tqdm

references = []
predictions = []

meteor_scores = []
rouge_scores = []

scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

model.eval()

for img_name in tqdm(references_dict.keys()):

    feature = torch.tensor(
        test_features[img_name]
    ).float().unsqueeze(0).to(device)

    pred = generate_caption(
        model,
        feature,
        vocab,
        idx2word
    )

    pred_tokens = pred.split()

    references.append(
        references_dict[img_name]
    )

    predictions.append(
        pred_tokens
    )

    meteor_scores.append(
        meteor_score(
            references_dict[img_name],
            pred_tokens
        )
    )

    gt_sentence = " ".join(
        references_dict[img_name][0]
    )

    rouge_scores.append(
        scorer.score(
            gt_sentence,
            pred
        )["rougeL"].fmeasure
    )

In [33]:
from nltk.translate.bleu_score import corpus_bleu

bleu1 = corpus_bleu(
    references,
    predictions,
    weights=(1,0,0,0)
)

bleu2 = corpus_bleu(
    references,
    predictions,
    weights=(0.5,0.5,0,0)
)

bleu3 = corpus_bleu(
    references,
    predictions,
    weights=(1/3,1/3,1/3,0)
)

bleu4 = corpus_bleu(
    references,
    predictions,
    weights=(0.25,0.25,0.25,0.25)
)

print("="*40)

print(f"BLEU-1  : {bleu1:.4f}")

print(f"BLEU-2  : {bleu2:.4f}")

print(f"BLEU-3  : {bleu3:.4f}")

print(f"BLEU-4  : {bleu4:.4f}")

print(f"METEOR  : {sum(meteor_scores)/len(meteor_scores):.4f}")

print(f"ROUGE-L : {sum(rouge_scores)/len(rouge_scores):.4f}")

BLEU-1  : 0.5845
BLEU-2  : 0.4117
BLEU-3  : 0.2790
BLEU-4  : 0.1883
METEOR  : 0.4012
ROUGE-L : 0.3450


In [ ]:
import pickle
import os

with open(
    os.path.join(MODEL_DIR, "idx2word.pkl"),
    "wb"
) as f:
    pickle.dump(idx2word, f)

print("idx2word saved.")

In [35]:
import pickle
import torch
import os

ckpt = torch.load(
    os.path.join(MODEL_DIR, "lstm_attn.pth"),
    map_location="cpu"
)

vocab = pickle.load(
    open(
        os.path.join(MODEL_DIR, "vocab.pkl"),
        "rb"
    )
)

idx2word = pickle.load(
    open(
        os.path.join(MODEL_DIR, "idx2word.pkl"),
        "rb"
    )
)

print("Embedding:", ckpt["embedding.weight"].shape)

print("Vocab:", len(vocab))

print("idx2word:", len(idx2word))

assert ckpt["embedding.weight"].shape[0] == len(vocab)
assert len(vocab) == len(idx2word)

print("✅ Everything matches.")

Embedding: torch.Size([4454, 256])
Vocab: 4454
idx2word: 4454
✅ Everything matches.
